### Google Drive Setup
This section connects the notebook environment to your Google Drive account, allowing the program to securely read your datasets and save outputs directly to your cloud storage.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

#Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


MessageError: Error: credential propagation was unsuccessful

### Importing Required Libraries
This block brings in all the necessary external tools and libraries required to run the machine learning pipeline. It imports packages for data manipulation (Pandas, Numpy), building the neural network (TensorFlow, Keras), splitting and evaluating data (Scikit-Learn), mathematical counting (Counter), and creating visual graphs (Matplotlib).

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.optimizers import Adam, SGD
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from sklearn.metrics import precision_score, recall_score
from collections import Counter
import matplotlib.pyplot as plt
import time as time


### Initialization and Data Loading
Here, the code locks the random seeds to guarantee that the mathematical operations yield the exact same results every time you run it. It also establishes the maximum length for text sequences and the maximum number of vocabulary words allowed before locating and loading the dataset from a CSV file.

In [ ]:
# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

MAX_LEN = 15
MAX_VOCAB_SIZE = 300

# ==== Moved CSV_FILEPATH definition here before its use ====
CSV_FILEPATH = '/content/drive/MyDrive/experimentation_data/ride-hailing-dataset-expanded.csv'

# Load CSV
df = pd.read_csv(CSV_FILEPATH)


### Text Tokenizer and Vocabulary Functions
This step defines two custom helper functions that allow the computer to understand human text. The `build_vocab` function reads through all the text to create a numerical "dictionary" of the most frequently used words. The `text_to_sequence` function uses that dictionary to convert sentences into lists of numbers, padding shorter sentences with zeros so that all sentences end up being the exact same length.

In [ ]:
# Basic Tokenizer & Vocabulary Builder
def build_vocab(texts, max_vocab_size):
    words = [word.lower() for text in texts for word in str(text).split()]
    word_counts = Counter(words)
    # Reserve index 0 for padding (<PAD>) and 1 for unknown tokens (<UNK>)
    most_common = word_counts.most_common(max_vocab_size - 2)
    vocab = {word: idx + 2 for idx, (word, _) in enumerate(most_common)}
    vocab['<PAD>'] = 0
    vocab['<UNK>'] = 1
    return vocab

def text_to_sequence(text, vocab, max_len):
    tokens = str(text).lower().split()
    seq = [vocab.get(token, 1) for token in tokens]
    if len(seq) < max_len:
        seq += [0] * (max_len - len(seq)) # Pad with 0
    else:
        seq = seq[:max_len]
    return seq


### Data Preparation and Splitting
This section applies the tokenizer functions defined above to the actual dataset, generating the final numerical input data (`X_data`) and the target categories (`y_data`). It then shuffles and divides the data into three separate batches: 70% for training the model, 15% for validating it as it learns, and 15% for final testing.

In [ ]:
# Build vocab and encode text
vocab = build_vocab(df['text'].tolist(), MAX_VOCAB_SIZE)
encoded_texts = [text_to_sequence(txt, vocab, MAX_LEN) for txt in df['text']]

X_data = np.array(encoded_texts)
y_data = np.array(df['label'].values)

num_classes = len(np.unique(y_data))

# Dataset Partitioning (70% Train, 15% Val, 15% Test)
X_train, X_temp, y_train, y_temp = train_test_split(X_data, y_data, test_size=0.30, random_state=42) # Corrected x_data to X_data
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)


### Hyperparameter Configuration
This block acts as the master control panel for the neural network. A Python dictionary named `CONFIG` is created to easily tune the model's settings, such as how fast it learns (`learning_rate`), how many rounds it trains (`epochs`), and how much it intentionally drops information to prevent memorizing the data (`dropout_rate`).

In [ ]:
# ====
# 1. LOAD & PREPROCESS CSV DATA
# ==
# CSV_FILEPATH definition was here, moved it up.
# Fixed sequence length (padded/truncated)
# Vocabulary size limit

# Truncate
# # #

# 2. HYPERPARAMETER CONFIGURATION

CONFIG = {
    "run_id": "EXP-34-B-9-KERAS-expanded dataset embed_dim down",
    "embed_dim": 16,
    "hidden_dim": 32,
    "learning_rate": 0.0005,
    "batch_size": 64,
    "epochs": 80,
    "optimizer_type": "Adam",
    "dropout_rate": 0.35
}


### Neural Network Architecture Construction
This builds the structure of the artificial intelligence model layer by layer. It starts with an `Embedding` layer to understand relationships between words, averages out the data across the sequence, processes it through a densely connected hidden layer to find complex patterns, applies a `Dropout` layer to prevent over-reliance on specific data points, and finishes with an output layer that assigns a final prediction category.

In [ ]:
#
# 3. KERAS SEQUENTIAL ARCHITECTURE

model = Sequential()

# Embedding Layer (Pillar 1: Architecture)
model.add(Embedding( # Corrected model. add to model.add and added opening parenthesis
    input_dim=len(vocab), # Corrected input_dim-len(vocab) to input_dim=len(vocab)
    output_dim=CONFIG["embed_dim"],
    # Removed input_length=MAX_LEN as it's deprecated and not needed with modern Keras for inference
    mask_zero=True # Added mask_zero for proper padding handling
)) # Added missing closing parenthesis

# Global Average Pooling across token sequence dimension
model.add(GlobalAveragePooling1D())

# First Hidden Layer
model.add(Dense(CONFIG["hidden_dim"], activation='relu'))

# Regularization Layer (Pillar 3: Regularization)
model.add(Dropout(CONFIG["dropout_rate"])) # Corrected space in Dropout

# Output Layer
model.add(Dense(num_classes, activation='softmax'))


### Model Optimizer and Compilation
Before training, the model needs instructions on how to evaluate its mistakes and adjust itself. This section selects the optimizer tool (Adam or SGD) based on the configurations, compiles the model with a loss function to calculate error, sets accuracy as the tracking metric, and finally prints a blueprint summary of the constructed model.

In [ ]:
                       # 4. OPTIMIZER & COMPILATION

if CONFIG["optimizer_type"] == "Adam":
    opt = Adam(learning_rate=CONFIG["learning_rate"])
else:
    opt = SGD(learning_rate=CONFIG["learning_rate"])

# Configure loss function, optimizer engine, and evaluation metrics
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=opt,
    metrics=['accuracy']
)

model.summary() # Corrected model.surmary() to model.summary()


### The Training and Evaluation Loop
This is the core operational phase where the actual machine learning occurs. The model loops over the training data, tracking progress against the validation set over a timed period. After completing its epochs, the code evaluates the network's predictive capabilities (calculating Precision, Recall, and an F1-Score) and prints a formatted log summarising the final performance statistics.

In [ ]:
# 5. THE CPERATIONAL LOOP (FIT & EVALUATE)

print(f" --- Starting Training Run: {CONFIG['run_id']} -- ")
start_time = time.time()
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=CONFIG["epochs"],
    batch_size=CONFIG["batch_size"],
    verbose=1
)

training_minutes = (time.time() - start_time) / 60
print(f"Training Time: {training_minutes:.4f} min")

# Compute final validation predictions for F1-Score calculation
val_preds_probs = model.predict(X_val)
val_preds = np.argmax(val_preds_probs, axis=1)

val_f1 = f1_score(y_val, val_preds, average='weighted')
val_precision = precision_score(y_val, val_preds, average='weighted', zero_division=0)
val_recall = recall_score(y_val, val_preds, average='weighted', zero_division=0)
print(f"Precision: {val_precision:.4f} | Recall: {val_recall:.4f}")
val_loss = history.history['val_loss'][-1]

# Final Log Output sumary for Excel logging
print("\n === EXP LDG ENTRY === ")
print(f"ID: {CONFIG['run_id']} | LR: {CONFIG['learning_rate']} | Hidden: {CONFIG['hidden_dim']} | Dropout: {CONFIG['dropout_rate']} | Final Val Loss: {val_loss:.4f} | Final Val F1: {val_f1:.4f}") # Removed spaces before colons


### Generating Performance Charts
The final block of code takes the historical learning data stored during training and visually charts it. It utilizes Matplotlib to plot a graph comparing the Training Loss with the Validation Loss side-by-side, which is highly useful to check visually if the model was effectively learning or simply memorizing (overfitting) the data.

In [ ]:
# 6. GENERATE LOSS CURVES CHART

train_losses = history.history['loss']
val_losses = history.history['val_loss']
epochs_range = range(1, CONFIG["epochs"] + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, train_losses, label='Training Loss', color='blue', linewidth=2, marker='o')
plt.plot(epochs_range, val_losses, label='Validation Loss', color='red', linewidth=2, linestyle='--', marker='s')

plt.title(f'Performance Monitoring: Loss Curves ({CONFIG["run_id"]})', fontsize=12, fontweight='bold')
plt.xlabel('Epochs', fontsize=10)
plt.ylabel('Loss (Sparse Categorical Cross-Entropy)', fontsize=10)
plt.xticks(epochs_range)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper right')
plt.tight_layout()

plt.show()
